In [ ]:
!pip install -q "trl>=0.19" peft accelerate datasets "bitsandbytes>=0.46.1"

In [ ]:
!nvidia-smi
!pip install -q -U "trl>=0.19" peft bitsandbytes accelerate datasets

In [ ]:
import os
import sys
import torch
from google.colab import drive

drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/oddo_sft"
sys.path.insert(0, DATA_DIR)
print(os.listdir(DATA_DIR))

BIG_GPU = torch.cuda.get_device_properties(0).total_memory > 20e9
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" if BIG_GPU else "Qwen/Qwen2.5-3B-Instruct"
# T4(7.5세대)는 bf16 계산을 못 한다 — 8.0 이상(L4·A100)에서만 bf16
BF16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float16
OUT_DIR = f"{DATA_DIR}/lora-{MODEL_ID.split('/')[-1]}-cbt"
print(torch.cuda.get_device_name(0), "|", MODEL_ID, "| bf16:", BF16)

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ds = load_dataset(
    "json",
    data_files={
        "train": f"{DATA_DIR}/cbt_train.jsonl",
        "validation": f"{DATA_DIR}/cbt_val.jsonl",
    },
)
print(ds)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=DTYPE,
        bnb_4bit_use_double_quant=True,
    ),
    device_map="auto",
    torch_dtype=DTYPE,
)

# 예시 길이 확인 — MAX_LEN보다 길면 completion이 잘리므로 MAX_LEN을 늘린다.
def n_tokens(row):
    text = tok.apply_chat_template(row["prompt"] + row["completion"], tokenize=False)
    return len(tok(text).input_ids)

lens = [n_tokens(r) for r in ds["train"]]
print("최대 토큰:", max(lens), "/ 평균:", sum(lens) // len(lens))

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MAX_LEN = 4096  # 셀 3의 최대 토큰보다 커야 한다

peft_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
cfg = SFTConfig(
    output_dir="/content/ckpt",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="no",
    max_length=MAX_LEN,
    completion_only_loss=True,
    gradient_checkpointing=True,
    bf16=False,   # T4: 정밀도 모드 끔 (L4·A100이면 bf16=True 가능)
    fp16=False,
    seed=42,
    report_to="none",
)
trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    processing_class=tok,
    peft_config=peft_cfg,
)

# 학습되는 LoRA 부분만 float32로 맞춘다 (bf16이 섞여 있으면 T4에서 에러)
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

trainer.train()
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("어댑터 저장 →", OUT_DIR)

In [ ]:
import json
import re
import statistics
import time

from counsel_output_check import check_reply

STAGE_KEYS = {
    0: {"missionDone", "missionFeedback"},
    1: {"emotionLabel", "intensity"},
    2: {"situation", "automaticThought"},
    3: {"insight", "beliefBefore"},
    4: {"evidenceFor", "evidenceAgainst", "alternativeThought", "beliefAfter"},
    5: {"mission", "sessionSummary"},
}
STAGE_RE = re.compile(r"\[현재 CBT 단계: (\d)")

model = trainer.model
model.eval()


def generate(prompt):
    text = tok.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors="pt").to(model.device)
    start = time.time()
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=300, do_sample=False)
    raw = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return raw, time.time() - start


def score(raw, prompt, ref):
    """서비스와 같은 기준 — JSON 그대로 파싱(코드블록 허용 안 함)."""
    system = prompt[0]["content"]
    stage = int(STAGE_RE.search(system).group(1))
    knows = "[지난 상담 주제]" in system or (
        "- 지난 미션: " in system and "- 지난 미션: 없음" not in system
    )
    users = [m["content"] for m in prompt if m["role"] == "user"]
    r = {"stage": stage, "json_ok": False, "key_ok": False, "rule_ok": False, "advance_match": False}
    try:
        data = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return r
    if not (
        isinstance(data, dict)
        and isinstance(data.get("reply"), str)
        and data["reply"].strip()
        and isinstance(data.get("advance"), bool)
        and isinstance(data.get("extracted", {}), dict)
    ):
        return r
    r["json_ok"] = True
    r["key_ok"] = set(data.get("extracted", {})) <= STAGE_KEYS[stage]
    r["violations"] = [v.code for v in check_reply(data["reply"], user_texts=users, has_recent_themes=knows)]
    r["rule_ok"] = not r["violations"]
    r["advance_match"] = data["advance"] == json.loads(ref)["advance"]
    return r


rows = []
for i, ex in enumerate(ds["validation"]):
    prompt, ref = ex["prompt"], ex["completion"][0]["content"]
    with model.disable_adapter():
        base_raw, base_sec = generate(prompt)
    sft_raw, sft_sec = generate(prompt)
    rows.append({
        "i": i,
        "ref": ref,
        "base": {"raw": base_raw, "sec": base_sec, **score(base_raw, prompt, ref)},
        "sft": {"raw": sft_raw, "sec": sft_sec, **score(sft_raw, prompt, ref)},
    })
    print(f"{i + 1}/{len(ds['validation'])}", end="\r")


def summary(name):
    rs = [r[name] for r in rows]
    n = len(rs)
    pct = lambda k: round(100 * sum(x[k] for x in rs) / n, 1)
    all_ok = round(100 * sum(x["json_ok"] and x["key_ok"] and x["rule_ok"] for x in rs) / n, 1)
    return {
        "n": n,
        "json_ok%": pct("json_ok"),
        "key_ok%": pct("key_ok"),
        "rule_ok%": pct("rule_ok"),
        "all_ok%": all_ok,
        "advance_match%": pct("advance_match"),
        "sec_median": round(statistics.median(x["sec"] for x in rs), 2),
    }


result = {
    "model": MODEL_ID,
    "lora": {"r": 16, "alpha": 32, "epochs": 3, "lr": 2e-4, "seed": 42},
    "train_examples": len(ds["train"]),
    "val_examples": len(ds["validation"]),
    "train_loss": [h["loss"] for h in trainer.state.log_history if "loss" in h],
    "eval_loss": [h["eval_loss"] for h in trainer.state.log_history if "eval_loss" in h],
    "base": summary("base"),
    "sft": summary("sft"),
}
print(json.dumps({k: result[k] for k in ("model", "base", "sft", "eval_loss")}, ensure_ascii=False, indent=2))

with open(f"{OUT_DIR}/eval.json", "w", encoding="utf-8") as f:
    json.dump({**result, "rows": rows}, f, ensure_ascii=False, indent=2)
print("평가 저장 →", f"{OUT_DIR}/eval.json")